# HW4: Metro Violations Agent — Full Experiment

**All-in-one notebook** for GPU server:
1. Setup & data generation
2. Baseline evaluation (Qwen 1.5B without training)
3. GRPO training with progress tracking
4. GRPO evaluation
5. Comparison & analysis plots

In [ ]:
!pip install -q torch transformers trl peft accelerate wandb matplotlib pandas tqdm

In [ ]:
import os, sys, json, time, random
import torch
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime, timedelta
from tqdm.auto import tqdm
from collections import defaultdict

sys.path.insert(0, '.')

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 1. Generate Data

In [ ]:
# Smoke test
!python scripts/generate_data.py --smoke

In [ ]:
# Full generation
!python scripts/generate_data.py

# Verify
for f in sorted(os.listdir('data')):
    if f.endswith('.jsonl'):
        n = sum(1 for _ in open(f'data/{f}', encoding='utf-8'))
        print(f'  {f}: {n} episodes')

## 2. Baseline Evaluation (Qwen 1.5B, no fine-tuning)

In [ ]:
from base.data import Data
from env.metro_env import MetroViolationsEnv
from verifier.trajectory_verifier import MetroTrajectoryVerifier
from agent.baseline_agent import BaselineAgent

def load_episodes(path, limit=None):
    eps = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                eps.append(Data.from_json(line.strip()))
                if limit and len(eps) >= limit:
                    break
    return eps

agent = BaselineAgent(model_name='Qwen/Qwen2.5-1.5B-Instruct', device='auto')
env = MetroViolationsEnv()
verifier = MetroTrajectoryVerifier()

In [ ]:
os.makedirs('logs', exist_ok=True)
baseline_results = {}

for bucket in range(1, 6):
    data_path = f'data/eval_d{bucket}.jsonl'
    if not os.path.exists(data_path):
        print(f'Skipping {data_path} (not found)')
        continue

    episodes = load_episodes(data_path)
    print(f'\n{"="*60}')
    print(f'Baseline eval_d{bucket}: {len(episodes)} episodes')
    print('=' * 60)

    all_metrics = []
    all_trajectories = []
    t0 = time.time()

    for i, ep in enumerate(tqdm(episodes, desc=f'eval_d{bucket}')):
        result = agent.run_episode(env, ep, verbose=False)
        metrics = verifier.verify_trajectory(env, ep, result['actions'])
        all_metrics.append({
            'question_id': ep.question_id,
            'difficulty': ep.difficulty,
            'task_type': ep.task_type,
            **metrics,
        })
        all_trajectories.append({
            **result,
            'metrics': {k: v for k, v in metrics.items() if k != 'info_trace'},
        })

    dt = time.time() - t0
    n = len(all_metrics)
    summary = {
        'model': 'baseline',
        'bucket': f'eval_d{bucket}',
        'total_episodes': n,
        'success_rate': sum(m['success'] for m in all_metrics) / n,
        'mean_reward': sum(m['total_reward'] for m in all_metrics) / n,
        'mean_steps': sum(m['steps'] for m in all_metrics) / n,
        'mean_tool_calls': sum(m['tool_calls'] for m in all_metrics) / n,
        'mean_policy_violations': sum(m['policy_violations'] for m in all_metrics) / n,
        'mean_invalid_actions': sum(m['invalid_actions'] for m in all_metrics) / n,
        'time_seconds': dt,
        'per_episode': all_metrics,
    }
    baseline_results[f'eval_d{bucket}'] = summary

    # Save
    with open(f'logs/metrics_baseline_eval_d{bucket}.json', 'w') as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    with open(f'logs/trajectories_baseline_eval_d{bucket}.jsonl', 'w') as f:
        for t in all_trajectories:
            f.write(json.dumps(t, ensure_ascii=False) + '\n')

    print(f'  Success: {summary["success_rate"]:.1%} | Reward: {summary["mean_reward"]:.3f} | '
          f'Steps: {summary["mean_steps"]:.1f} | Tools: {summary["mean_tool_calls"]:.1f} | '
          f'Violations: {summary["mean_policy_violations"]:.2f} | Time: {dt:.0f}s')

In [ ]:
# Baseline summary table
print(f'{"Bucket":<12} {"Success":>10} {"Reward":>10} {"Steps":>8} {"Tools":>8} {"Violations":>12}')
print('-' * 62)
for b, s in sorted(baseline_results.items()):
    print(f'{b:<12} {s["success_rate"]:>9.1%} {s["mean_reward"]:>10.3f} '
          f'{s["mean_steps"]:>8.1f} {s["mean_tool_calls"]:>8.1f} {s["mean_policy_violations"]:>12.2f}')

## 3. GRPO Training

In [ ]:
# Free baseline model memory
del agent
torch.cuda.empty_cache()

In [ ]:
from training.grpo_train import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    model_name='Qwen/Qwen2.5-1.5B-Instruct',
    output_dir='checkpoints/grpo',
    num_epochs=2,
    batch_size=2,
    num_generations=4,
    learning_rate=5e-6,
    use_lora=True,
    lora_r=16,
    lora_alpha=32,
    log_dir='logs/training',
    log_every=5,
    eval_every=50,
    save_every=100,
    use_wandb=False,  # set True if wandb configured
    eval_limit=10,
    max_steps_per_episode=15,
)

trainer = GRPOTrainer(config)
trainer.setup_model()
trainer.load_data()

In [ ]:
# === Training loop with progress tracking ===

train_logs = []
eval_logs = []
global_step = 0
total_batches = (len(trainer.train_data) // config.batch_size) * config.num_epochs

print(f'Total training batches: {total_batches}')
print(f'Episodes: {len(trainer.train_data)}, Batch size: {config.batch_size}, '
      f'Generations: {config.num_generations}, Epochs: {config.num_epochs}')
print()

pbar = tqdm(total=total_batches, desc='GRPO Training', unit='batch')

for epoch in range(config.num_epochs):
    random.shuffle(trainer.train_data)
    epoch_metrics = defaultdict(list)

    for i in range(0, len(trainer.train_data), config.batch_size):
        batch = trainer.train_data[i:i + config.batch_size]
        if len(batch) < config.batch_size:
            continue

        t0 = time.time()
        metrics = trainer.train_step(batch)
        dt = time.time() - t0
        global_step += 1

        log = {'step': global_step, 'epoch': epoch+1, 'time': dt, **metrics}
        train_logs.append(log)
        for k, v in metrics.items():
            epoch_metrics[k].append(v)

        # Update progress bar
        pbar.update(1)
        pbar.set_postfix({
            'loss': f'{metrics["loss"]:.4f}',
            'reward': f'{metrics["mean_reward"]:.3f}',
            'success': f'{metrics["success_rate"]:.0%}',
            'eta': str(timedelta(seconds=int(dt * (total_batches - global_step)))),
        })

        # Save training log
        with open(os.path.join(config.log_dir, 'train_log.jsonl'), 'a') as f:
            f.write(json.dumps(log) + '\n')

        # Periodic eval
        if global_step % config.eval_every == 0:
            print(f'\n--- Eval at step {global_step} ---')
            eval_result = trainer.evaluate()
            eval_logs.append({'step': global_step, 'results': eval_result})
            with open(os.path.join(config.log_dir, 'eval_log.jsonl'), 'a') as f:
                f.write(json.dumps({'step': global_step, **eval_result}) + '\n')

        # Periodic save
        if global_step % config.save_every == 0:
            trainer.save_checkpoint(f'{config.output_dir}/step_{global_step}')

    # Epoch summary
    print(f'\n=== Epoch {epoch+1} Summary ===')
    for k in ['loss', 'mean_reward', 'success_rate', 'mean_steps']:
        vals = epoch_metrics.get(k, [])
        if vals:
            print(f'  {k}: {sum(vals)/len(vals):.4f}')

pbar.close()

# Final save
trainer.save_checkpoint(f'{config.output_dir}/final')
print(f'\nTraining complete! Final model: {config.output_dir}/final')

## 4. Training Curves

In [ ]:
df = pd.DataFrame(train_logs)
if df.empty:
    # Try loading from file
    log_path = os.path.join(config.log_dir, 'train_log.jsonl')
    if os.path.exists(log_path):
        df = pd.DataFrame([json.loads(l) for l in open(log_path)])

if not df.empty:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('GRPO Training Curves', fontsize=16, fontweight='bold')
    plots = [
        ('loss', 'Loss', 'tab:red'),
        ('mean_reward', 'Mean Reward', 'tab:blue'),
        ('success_rate', 'Success Rate', 'tab:green'),
        ('mean_steps', 'Mean Steps', 'tab:orange'),
        ('mean_tools', 'Mean Tool Calls', 'tab:purple'),
        ('mean_violations', 'Policy Violations', 'tab:brown'),
    ]
    w = max(1, len(df) // 30)
    for ax, (key, title, color) in zip(axes.flatten(), plots):
        if key in df.columns:
            sm = df[key].rolling(window=w, min_periods=1).mean()
            ax.plot(df['step'], df[key], color=color, alpha=0.15)
            ax.plot(df['step'], sm, color=color, linewidth=2, label='smoothed')
            ax.set_title(title, fontweight='bold')
            ax.set_xlabel('Step')
            ax.grid(alpha=0.3)
            ax.legend()
    plt.tight_layout()
    plt.savefig('logs/training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No training logs found')

## 5. GRPO Evaluation

In [ ]:
# Load trained model as agent
del trainer
torch.cuda.empty_cache()

grpo_agent = BaselineAgent(
    model_name='checkpoints/grpo/final',
    device='auto',
)
env = MetroViolationsEnv()
verifier = MetroTrajectoryVerifier()

grpo_results = {}
for bucket in range(1, 6):
    data_path = f'data/eval_d{bucket}.jsonl'
    if not os.path.exists(data_path):
        continue
    episodes = load_episodes(data_path)
    print(f'\n{"="*60}')
    print(f'GRPO eval_d{bucket}: {len(episodes)} episodes')

    all_metrics = []
    all_trajectories = []
    t0 = time.time()

    for ep in tqdm(episodes, desc=f'eval_d{bucket}'):
        result = grpo_agent.run_episode(env, ep)
        metrics = verifier.verify_trajectory(env, ep, result['actions'])
        all_metrics.append({'question_id': ep.question_id, 'difficulty': ep.difficulty,
                            'task_type': ep.task_type, **metrics})
        all_trajectories.append({**result, 'metrics': {k:v for k,v in metrics.items() if k!='info_trace'}})

    dt = time.time() - t0
    n = len(all_metrics)
    summary = {
        'model': 'grpo', 'bucket': f'eval_d{bucket}', 'total_episodes': n,
        'success_rate': sum(m['success'] for m in all_metrics) / n,
        'mean_reward': sum(m['total_reward'] for m in all_metrics) / n,
        'mean_steps': sum(m['steps'] for m in all_metrics) / n,
        'mean_tool_calls': sum(m['tool_calls'] for m in all_metrics) / n,
        'mean_policy_violations': sum(m['policy_violations'] for m in all_metrics) / n,
        'time_seconds': dt, 'per_episode': all_metrics,
    }
    grpo_results[f'eval_d{bucket}'] = summary
    with open(f'logs/metrics_grpo_eval_d{bucket}.json', 'w') as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)
    with open(f'logs/trajectories_grpo_eval_d{bucket}.jsonl', 'w') as f:
        for t in all_trajectories:
            f.write(json.dumps(t, ensure_ascii=False) + '\n')
    print(f'  Success: {summary["success_rate"]:.1%} | Reward: {summary["mean_reward"]:.3f} | '
          f'Steps: {summary["mean_steps"]:.1f} | Time: {dt:.0f}s')

## 6. Comparison: Baseline vs GRPO

In [ ]:
# Load results if not in memory
if not baseline_results:
    for b in range(1,6):
        p = f'logs/metrics_baseline_eval_d{b}.json'
        if os.path.exists(p):
            baseline_results[f'eval_d{b}'] = json.load(open(p))
if not grpo_results:
    for b in range(1,6):
        p = f'logs/metrics_grpo_eval_d{b}.json'
        if os.path.exists(p):
            grpo_results[f'eval_d{b}'] = json.load(open(p))

# Table
print(f'{"Bucket":<10} {"":>6} {"Success":>10} {"Reward":>10} {"Steps":>8} {"Tools":>8} {"Violations":>12}')
print('=' * 66)
for b in sorted(set(list(baseline_results) + list(grpo_results))):
    bl = baseline_results.get(b, {})
    gr = grpo_results.get(b, {})
    print(f'{b:<10} {"base":>6} {bl.get("success_rate",0):>9.1%} {bl.get("mean_reward",0):>10.3f} '
          f'{bl.get("mean_steps",0):>8.1f} {bl.get("mean_tool_calls",0):>8.1f} {bl.get("mean_policy_violations",0):>12.2f}')
    print(f'{"":<10} {"grpo":>6} {gr.get("success_rate",0):>9.1%} {gr.get("mean_reward",0):>10.3f} '
          f'{gr.get("mean_steps",0):>8.1f} {gr.get("mean_tool_calls",0):>8.1f} {gr.get("mean_policy_violations",0):>12.2f}')
    d_sr = gr.get('success_rate',0) - bl.get('success_rate',0)
    print(f'{"":<10} {"delta":>6} {d_sr:>+9.1%}\n')

In [ ]:
# === ALL COMPARISON PLOTS ===
buckets = sorted(set(list(baseline_results) + list(grpo_results)))
labels = [b.replace('eval_','').upper() for b in buckets]
x = range(len(buckets))
w = 0.35

metrics_list = [
    ('success_rate', 'Success Rate', True),
    ('mean_reward', 'Mean Reward', False),
    ('mean_steps', 'Mean Steps', False),
    ('mean_tool_calls', 'Mean Tool Calls', False),
    ('mean_policy_violations', 'Mean Policy Violations', False),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Baseline vs GRPO — All Eval Buckets', fontsize=16, fontweight='bold')

for ax, (key, title, is_pct) in zip(axes.flatten(), metrics_list):
    bv = [baseline_results.get(b,{}).get(key,0) for b in buckets]
    gv = [grpo_results.get(b,{}).get(key,0) for b in buckets]
    ax.bar([i-w/2 for i in x], bv, w, label='Baseline', color='#5B8DEF', alpha=0.85)
    ax.bar([i+w/2 for i in x], gv, w, label='GRPO', color='#FF6B6B', alpha=0.85)
    ax.set_title(title, fontweight='bold')
    ax.set_xticks(list(x)); ax.set_xticklabels(labels)
    ax.legend(); ax.grid(axis='y', alpha=0.3)
    if is_pct:
        ax.set_ylim(0, 1.05)
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'{v:.0%}'))

axes.flatten()[-1].set_visible(False)
plt.tight_layout()
os.makedirs('logs/plots', exist_ok=True)
plt.savefig('logs/plots/comparison_bars.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === Per-difficulty success rate line plot ===
fig, ax = plt.subplots(figsize=(10, 6))

for model_name, results, color, marker in [('Baseline', baseline_results, '#5B8DEF', 'o'),
                                             ('GRPO', grpo_results, '#FF6B6B', 's')]:
    xs, ys = [], []
    for b in sorted(results.keys()):
        per_ep = results[b].get('per_episode', [])
        by_diff = defaultdict(list)
        for ep in per_ep:
            by_diff[ep['difficulty']].append(ep['success'])
        for d in sorted(by_diff):
            xs.append(d)
            ys.append(sum(by_diff[d]) / len(by_diff[d]))
    if xs:
        ax.plot(xs, ys, color=color, marker=marker, linewidth=2, markersize=8, label=model_name)

ax.set_xlabel('Difficulty', fontsize=12)
ax.set_ylabel('Success Rate', fontsize=12)
ax.set_title('Success Rate by Difficulty Level', fontsize=14, fontweight='bold')
ax.legend(fontsize=12); ax.grid(alpha=0.3)
ax.set_xticks(range(1, 11))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_: f'{v:.0%}'))
plt.savefig('logs/plots/success_by_difficulty.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === Task type breakdown ===
for model_name, results in [('Baseline', baseline_results), ('GRPO', grpo_results)]:
    by_type = defaultdict(lambda: {'total': 0, 'success': 0})
    for b, data in results.items():
        for ep in data.get('per_episode', []):
            tt = ep['task_type']
            by_type[tt]['total'] += 1
            by_type[tt]['success'] += int(ep['success'])
    print(f'\n{model_name} — Success by task type:')
    for tt in sorted(by_type):
        s = by_type[tt]
        rate = s['success'] / s['total'] if s['total'] else 0
        print(f'  {tt:<25} {rate:>6.1%}  ({s["success"]}/{s["total"]})')

## 7. Failure Analysis & Proxy Reward Critique

In [ ]:
# Show example failure trajectories from GRPO
for bucket in range(1, 6):
    path = f'logs/trajectories_grpo_eval_d{bucket}.jsonl'
    if not os.path.exists(path): continue
    failures = []
    with open(path) as f:
        for line in f:
            e = json.loads(line)
            if not e.get('metrics',{}).get('success', False):
                failures.append(e)
    if failures:
        ex = failures[0]
        print(f'\n=== eval_d{bucket} failure example (d={ex["difficulty"]}, {ex["task_type"]}) ===')
        for i, a in enumerate(ex['actions'][:6]):
            print(f'  Step {i+1}: {a[:150]}')
        print(f'  Metrics: {ex["metrics"]}')

### Proxy Reward Failure Modes

1. **Early FINAL_ANSWER hacking**: Step penalty (-0.02) incentivizes guessing fast → fix: increase outcome weight
2. **Avoiding open_case**: Confirmation penalty risk makes model avoid state-changes even when required → fix: bonus for correct case opening
3. **Always-no on d8**: Model learns to answer 'no' without querying for empty-result tasks → fix: minimum tool call requirement
4. **SQL memorization**: Patterns that work on training seeds fail on new data → fix: more diverse seeds
5. **Repeated query gaming**: Slightly modified SQL to avoid repeat penalty → fix: normalize SQL before comparison

In [ ]:
print('All results saved in logs/ directory')
print('Plots saved in logs/plots/')
print('Model checkpoints in checkpoints/grpo/')